In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tensor-Netzwerk-Fehlerminderung (TEM): Eine Qiskit Function von Algorithmiq
> **Note:** Qiskit Functions sind ein experimentelles Feature, das ausschließlich Nutzerinnen und Nutzern des IBM Quantum&reg; Premium Plan, Flex Plan und On-Prem (über die IBM Quantum Platform API) zur Verfügung steht. Sie befinden sich im Preview-Status und können sich noch ändern.


<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese oder neuere Versionen zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-catalog~=0.11.0
```
</details>
## Überblick
Die Tensor-Netzwerk-Fehlerminderungsmethode (TEM) von Algorithmiq ist ein hybrides
quanten-klassisches Verfahren, das Rauschminderung vollständig in der klassischen
Nachverarbeitungsphase durchführt. Mit TEM kannst du Erwartungswerte von Observablen berechnen
und dabei die unvermeidlichen rauschbedingten Fehler auf Quantenhardware mit
verbesserter Genauigkeit und Kosteneffizienz mindern – eine attraktive Option
für Quantenforscher und Industriepraktiker gleichermaßen.

Die Methode besteht darin, ein Tensor-Netzwerk zu konstruieren, das die Inverse des
globalen Rauschkanals darstellt, der den Zustand des Quantenprozessors beeinflusst,
und diese Abbildung dann auf informationsvollständige Messergebnisse anzuwenden,
die aus dem verrauschten Zustand gewonnen wurden, um unverzerrte Schätzer für die
Observablen zu erhalten.

Als Vorteil nutzt TEM informationsvollständige Messungen, um Zugang zu einer
großen Menge gemilderter Erwartungswerte von Observablen zu erhalten, und hat
optimalen Sampling-Overhead auf der Quantenhardware, wie in Filippov et
al. (2023), [arXiv:2307.11740](https://arxiv.org/abs/2307.11740), und Filippov
et al. (2024), [arXiv:2403.13542](https://arxiv.org/abs/2403.13542) beschrieben. Der
Mess-Overhead bezieht sich auf die Anzahl zusätzlicher Messungen, die für eine
effiziente Fehlerminderung erforderlich sind – ein entscheidender Faktor für die
Durchführbarkeit von Quantenberechnungen. Daher hat TEM das Potenzial, einen
Quantenvorteil in komplexen Szenarien zu ermöglichen, etwa bei Anwendungen in den
Bereichen Quantenchaos, Vielteilchenphysik, Hubbard-Dynamik und Simulationen
kleiner Molekülchemie.

Die wichtigsten Merkmale und Vorteile von TEM lassen sich wie folgt zusammenfassen:

1.  **Optimaler Mess-Overhead**: TEM ist optimal hinsichtlich der
[theoretischen Schranken](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.131.210601),
d. h. keine andere Methode kann einen geringeren Mess-Overhead erzielen. Anders
gesagt benötigt TEM die minimale Anzahl zusätzlicher Messungen für die
Fehlerminderung. Das bedeutet wiederum, dass TEM die Quantenlaufzeit minimiert.
2.  **Kosteneffizienz**: Da TEM die Rauschminderung vollständig in der
Nachverarbeitungsphase übernimmt, müssen dem Quantencomputer keine zusätzlichen
Circuits hinzugefügt werden. Das macht die Berechnung nicht nur günstiger,
sondern verringert auch das Risiko, durch die Unvollkommenheiten von
Quantengeräten zusätzliche Fehler einzuführen.
3.  **Schätzung mehrerer Observablen**: Dank informationsvollständiger
Messungen schätzt TEM effizient mehrere Observablen mit denselben
Messdaten vom Quantencomputer.
4.  **Messfehlerminderung**: Die TEM Qiskit Function enthält außerdem eine
[proprietäre Methode zur Messfehlerminderung](https://journals.aps.org/prresearch/abstract/10.1103/PhysRevResearch.5.033154),
die Auslesfehler nach einem kurzen Kalibrierungslauf deutlich reduzieren kann.
5.  **Genauigkeit**: TEM verbessert die Genauigkeit und Zuverlässigkeit
digitaler Quantensimulationen erheblich und macht Quantenalgorithmen präziser
und verlässlicher.
## Beschreibung
Die TEM-Funktion ermöglicht es dir, fehlergeminderte Erwartungswerte für
mehrere Observablen eines Quantum Circuits mit minimalem Sampling-Overhead zu berechnen.
Der Circuit wird mit einer informationsvollständigen positiven operatorwertigen
Maßnahme (IC-POVM) gemessen, und die gesammelten Messergebnisse werden auf einem
klassischen Computer verarbeitet. Diese Messung wird verwendet, um die
Tensor-Netzwerk-Methoden durchzuführen und eine rausch-invertierende Abbildung
aufzubauen. Die Funktion wendet eine Abbildung an, die den gesamten verrauschten
Circuit vollständig invertiert, indem Tensor-Netzwerke zur Darstellung der
verrauschten Schichten verwendet werden.

![TEM-Schemata](../docs/images/guides/algorithmiq-tem/tem_scheme.svg "Fehlergeminderte Schätzung einer Observablen O durch Nachverarbeitung der Messergebnisse des verrauschten Quantenprozessors. U und N bezeichnen eine ideale Quantenoperation und die zugehörige Rauschkarte, die im Allgemeinen nicht-lokal sein kann (und auf graue Boxen ausgedehnt wird). D steht für einen Tensor von Operatoren, die dual zu den Effekten der IC-Messung sind. Das Rauschminderungsmodul M ist ein Tensor-Netzwerk, das effizient von der Mitte nach außen kontrahiert wird. Die erste Iteration der Kontraktion wird durch die gepunktete lila Linie dargestellt, die zweite durch die gestrichelte Linie und die dritte durch die durchgezogene Linie.")

Sobald die Circuits an die Funktion übermittelt werden, werden sie transpiliert und
optimiert, um die Anzahl der Schichten mit Zwei-Qubit-Gates zu minimieren (den
rauschintensiveren Gates auf Quantengeräten). Das die Schichten beeinflussende
Rauschen wird über
[Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/noise-learner-noise-learner)
mithilfe eines spärlichen Pauli-Lindblad-Rauschmodells erlernt, wie in E. van den Berg, Z.
Minev, A. Kandala, K. Temme, Nat. Phys. (2023) beschrieben.
[arXiv:2201.09866](https://arxiv.org/abs/2201.09866).

Das Rauschmodell ist eine genaue Beschreibung des Rauschens auf dem Gerät und kann
subtile Merkmale erfassen, einschließlich Qubit-Übersprechen. Allerdings kann das
Rauschen auf den Geräten schwanken und driften, sodass das erlernte Rauschmodell
zum Zeitpunkt der Schätzung möglicherweise nicht mehr akkurat ist. Das kann zu
ungenauen Ergebnissen führen.
## Erste Schritte
Authentifiziere dich mit deinem [IBM Quantum Platform API-Schlüssel](http://quantum.cloud.ibm.com/) und wähle die TEM-Funktion wie folgt aus. (Dieser Codeausschnitt setzt voraus, dass du dein Konto bereits [in deiner lokalen Umgebung gespeichert hast](/guides/functions#install-qiskit-functions-catalog-client).)

In [1]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

tem_function_name = "algorithmiq/tem"

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Load your function
tem = catalog.load(tem_function_name)

## Beispiel
Der folgende Codeausschnitt zeigt ein Beispiel, bei dem TEM verwendet wird, um die Erwartungswerte einer Observablen für einen einfachen Quantum Circuit zu berechnen.

In [2]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp

# Create a quantum circuit
qc = QuantumCircuit(3)
qc.u(0.4, 0.9, -0.3, 0)
qc.u(-0.4, 0.2, 1.3, 1)
qc.u(-1.2, -1.2, 0.3, 2)
for _ in range(2):
    qc.barrier()
    qc.cx(0, 1)
    qc.cx(2, 1)
    qc.barrier()
    qc.u(0.4, 0.9, -0.3, 0)
    qc.u(-0.4, 0.2, 1.3, 1)
    qc.u(-1.2, -1.2, 0.3, 2)

# Define the observables
observable = SparsePauliOp("IYX", 1.0)

# Define the execution options
pub = (qc, [observable])
options = {"default_precision": 0.02}

# Define backend to use. TEM will choose the least-busy device reported by IBM if not specified
backend_name = "ibm_torino"

job = tem.run(pubs=[pub], backend_name=backend_name, options=options)

Verwende die Qiskit Serverless APIs, um den Status deines Qiskit Function-Workloads zu überprüfen:

In [3]:
print(job.status())

QUEUED


You can return the results as:

In [4]:
result = job.result()
evs = result[0].data.evs

Du kannst die Ergebnisse wie folgt abrufen:

In [5]:
print(job.result())

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(1,), dtype=float64>), stds=np.ndarray(<shape=(1,), dtype=float64>)), metadata={'evs_non_mitigated': array([-0.06314623]), 'stds_non_mitigated': array([0.05917147]), 'evs_mitigated_no_readout_mitigation': array([-0.06411205]), 'stds_mitigated_no_readout_mitigation': array([0.05992467]), 'evs_non_mitigated_with_readout_mitigation': array([-0.07028881]), 'stds_non_mitigated_with_readout_mitigation': array([0.06353934])})], metadata={'resource_usage': {'RUNNING: OPTIMIZING_FOR_HARDWARE': {'CPU_TIME': 0.915754}, 'RUNNING: WAITING_FOR_QPU': {'CPU_TIME': 18.804865}, 'RUNNING: POST_PROCESSING': {'CPU_TIME': 10.433445}, 'RUNNING: EXECUTING_QPU': {'QPU_TIME': 159.0}}})


> **Info:** Der erwartete Wert für den rauschfreien Circuit für den angegebenen Operator sollte etwa `0.18409094298943401` betragen.
## Eingaben
**Parameter**

Name | Typ | Beschreibung | Erforderlich | Standard | Beispiel
-- | -- | -- | -- | -- | --
`pubs` | Iterable[EstimatorPubLike] | Ein iterierbares Objekt aus PUB-ähnlichen (primitive unified bloc) Objekten, z. B. Tupeln `(circuit, observables)` oder `(circuit, observables, parameters, precision)`. Weitere Informationen findest du unter [Überblick über PUBs](/guides/primitive-input-output#overview-of-pubs). Wird ein Nicht-ISA-Circuit übergeben, wird er mit optimalen Einstellungen transpiliert. Bei einem ISA-Circuit wird keine Transpilierung durchgeführt; in diesem Fall muss die Observable auf dem gesamten QPU definiert sein. | Ja | k. A. | (circuit, observables)
`backend_name` | str | Name des Backends, an das die Anfrage gesendet wird. | Nein | Wenn nicht angegeben, wird das am wenigsten ausgelastete Backend verwendet. | "ibm_fez"
`options` | dict | Eingabeoptionen. Weitere Details im Abschnitt `Options`. | Nein | Weitere Details im Abschnitt `Options`. | {"max_bond_dimension": 100\}

> **Caution:** TEM hat derzeit folgende Einschränkungen:
> 
>   - Parametrisierte Circuits werden nicht unterstützt. Das parameters-Argument sollte auf `None` gesetzt werden, wenn precision angegeben ist. Diese Einschränkung wird in zukünftigen Versionen aufgehoben.
>   - Nur Circuits ohne Schleifen werden unterstützt. Diese Einschränkung wird in zukünftigen Versionen aufgehoben.
>   - Nicht-unitäre Gates wie Reset, Measure und alle Formen von Kontrollfluss werden nicht unterstützt. Unterstützung für Reset wird in kommenden Releases hinzugefügt.
### Optionen
Ein Dictionary mit den erweiterten Optionen für TEM. Das Dictionary kann die Schlüssel aus der folgenden Tabelle enthalten. Wenn eine Option nicht angegeben wird, wird der in der Tabelle aufgeführte Standardwert verwendet. Die Standardwerte eignen sich gut für die typische Nutzung von TEM.

Name | Auswahlmöglichkeiten | Beschreibung | Standard
-- | -- | -- | --
`tem_max_bond_dimension` | int | Die maximale Bond-Dimension, die für die Tensor-Netzwerke verwendet wird. | 500 |
`tem_compression_cutoff` | float | Der Cutoff-Wert für die Tensor-Netzwerke. | 1e-16
`compute_shadows_bias_from_observable` | bool | Ein boolesches Flag, das angibt, ob der Bias für das klassische Shadows-Messprotokoll auf die PUB-Observable zugeschnitten werden soll. Bei False wird das klassische Shadows-Protokoll verwendet (gleiche Wahrscheinlichkeit für Z, X, Y). | False
`shadows_bias` | np.ndarray | Der Bias für das randomisierte klassische Shadows-Messprotokoll; ein 1D- oder 2D-Array der Größe 3 bzw. der Form (num_qubits, 3). Reihenfolge: ZXY | np.array([1 / 3, 1 / 3, 1 / 3])
`max_execution_time` | int oder `None` | Die maximale Ausführungszeit auf dem QPU in Sekunden. Wenn die Laufzeit diesen Wert überschreitet, wird der Job abgebrochen. Bei `None` gilt ein von Qiskit Runtime gesetztes Standardlimit. | `None`
`num_randomizations` | int | Die Anzahl der Randomisierungen für das Rauschlernen und Gate-Twirling. | 32
`max_layers_to_learn` | int | Die maximale Anzahl eindeutiger Schichten, die erlernt werden sollen. | 4
`mitigate_readout_error` | bool | Ein boolesches Flag, das angibt, ob Auslesefehlerminderung durchgeführt werden soll. | True
`num_readout_calibration_shots` | int | Die Anzahl der Shots für die Auslesefehlerminderung. | 10000
`default_precision` | float | Die Standardpräzision für PUBs, für die keine Präzision angegeben ist. | 0.02
`seed` | int oder `None` | Setzt den Seed des Zufallszahlengenerators für Reproduzierbarkeit. Bei `None` wird kein Seed gesetzt. | None
## Ausgaben
Ein Qiskit [PrimitiveResults](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PrimitiveResult)-Objekt, das das TEM-geminderte Ergebnis enthält. Das Ergebnis für jeden PUB wird als [PubResult](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.PubResult) mit den folgenden Feldern zurückgegeben:

Name | Typ | Beschreibung
-- | -- | --
data | DataBin | Ein Qiskit [DataBin](https://docs.quantum.ibm.com/api/qiskit/qiskit.primitives.DataBin), das die TEM-geminderte Observable und ihren Standardfehler enthält. Das DataBin hat folgende Felder: <ul><li>`evs`: Der TEM-geminderte Observablenwert.</li><li>`stds`: Der Standardfehler der TEM-geminderten Observablen.</li></ul>
metadata | dict | Ein Dictionary mit zusätzlichen Ergebnissen. Das Dictionary enthält folgende Schlüssel: <ul><li>`"evs_non_mitigated"`: Der Observablenwert ohne Fehlerminderung.</li><li>`"stds_non_mitigated"`: Der Standardfehler des Ergebnisses ohne Fehlerminderung.</li><li>`"evs_mitigated_no_readout_mitigation"`: Der Observablenwert mit Fehlerminderung, aber ohne Auslesefehlerminderung.</li><li>`"stds_mitigated_no_readout_mitigation"`: Der Standardfehler des Ergebnisses mit Fehlerminderung, aber ohne Auslesefehlerminderung.</li><li>`"evs_non_mitigated_with_readout_mitigation"`: Der Observablenwert ohne Fehlerminderung, aber mit Auslesefehlerminderung.</li><li>`"stds_non_mitigated_with_readout_mitigation"`: Der Standardfehler des Ergebnisses ohne Fehlerminderung, aber mit Auslesefehlerminderung.</li></ul>
## Fehlermeldungen abrufen
Wenn der Status deines Workloads ERROR ist, verwende job.result(), um die Fehlermeldung wie folgt abzurufen: